In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [2]:
df = pd.read_pickle('G:/Mohanraj D_OFFICIAL/GUVI (Data Analyst)/Mini-Projects/PatrolIQ/cleaned_chicago_crime_dataset.pkl')
df.head()

,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,10819224,JA119772,2016-12-31 23:59:00,100XX W OHARE ST,0810,THEFT,OVER $500,AIRPORT BUILDING NON-TERMINAL - SECURE AREA,False,False,...,41,76,06,1100658.0,1934241.0,2016,2018-02-10 15:50:01,41.976290,-87.905227,"(41.976290414, -87.905227221)"
1,10801137,JA100016,2016-12-31 23:58:00,0000X W 113TH PL,0430,BATTERY,AGGRAVATED: OTHER DANG WEAPON,RESIDENCE,False,False,...,34,49,04B,1178014.0,1829709.0,2016,2018-02-10 15:50:01,41.688033,-87.623931,"(41.688033246, -87.623931468)"
2,10801110,JA100027,2016-12-31 23:55:00,030XX N LINCOLN AVE,2250,LIQUOR LAW VIOLATION,LIQUOR LICENSE VIOLATION,RESIDENCE,True,False,...,32,6,22,1166154.0,1920300.0,2016,2017-01-07 15:56:13,41.936885,-87.664770,"(41.936884881, -87.66476981)"
3,10802006,JA100012,2016-12-31 23:55:00,0000X E WACKER PL,0486,BATTERY,DOMESTIC BATTERY SIMPLE,HOTEL/MOTEL,False,True,...,42,32,08B,1176964.0,1902140.0,2016,2018-02-10 15:50:01,41.886815,-87.625593,"(41.886814897, -87.625592678)"
4,10801865,JA100839,2016-12-31 23:54:00,078XX S INDIANA AVE,1310,CRIMINAL DAMAGE,TO PROPERTY,RESIDENCE,False,True,...,6,69,14,1178949.0,1853139.0,2016,2018-02-10 15:50:01,41.752307,-87.619798,"(41.752307019, -87.619797619)"


In [3]:
pd.set_option('display.max_columns', None)

In [4]:
# Drop unnecessary columns for clustering
df = df.drop(columns=['Case Number','Location', 'ID', 'Updated On', 'Block', 'Description', 'IUCR', 'FBI Code'])

In [5]:
# Since X co-oedinate is highly co-related to Longitude, we romove X-coordinate column. 
# Also, Y co-oedinate is highly co-related to Latitude, we romove Y-coordinate column. 
df = df.drop(columns=['X Coordinate', 'Y Coordinate'])

In [6]:
# # Create temporal features
df["Month"] = df["Date"].dt.month
df["Day_of_Week"] = df["Date"].dt.day_name()
df["Hour"] = df["Date"].dt.hour
df["Is_Weekend"] = df["Day_of_Week"].isin(["Saturday", "Sunday"]).astype(int)

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

df["Season"] = df["Month"].apply(get_season)

In [7]:
# geographical features
# district alreay in the data

# coordinate binning
df["Lat_bin"] = pd.cut(df["Latitude"], bins=20)
df["Lon_bin"] = pd.cut(df["Longitude"], bins=20)

In [8]:
# creating crime severity column
severity_map = {
    "HOMICIDE": 5,
    "CRIMINAL SEXUAL ASSAULT": 5,
    "KIDNAPPING": 5,

    "ROBBERY": 4,
    "WEAPONS VIOLATION": 4,
    "ARSON": 4,
    "HUMAN TRAFFICKING": 4,

    "ASSAULT": 3,
    "BATTERY": 3,
    "SEX OFFENSE": 3,
    "STALKING": 3,
    "INTIMIDATION": 3,
    "OFFENSE INVOLVING CHILDREN": 3,

    "BURGLARY": 2,
    "MOTOR VEHICLE THEFT": 2,
    "CRIMINAL DAMAGE": 2,
    "CRIMINAL TRESPASS": 2,
    "DECEPTIVE PRACTICE": 2,

    "THEFT": 1,
    "NARCOTICS": 1,
    "PROSTITUTION": 1,
    "PUBLIC PEACE VIOLATION": 1,
    "LIQUOR LAW VIOLATION": 1,
    "GAMBLING": 1,
    "OBSCENITY": 1,
    "NON-CRIMINAL": 1,
    "OTHER OFFENSE": 1,
    "OTHER NARCOTIC VIOLATION": 1,
    "INTERFERENCE WITH PUBLIC OFFICER": 1,
    "PUBLIC INDECENCY": 1,
    "CONCEALED CARRY LICENSE VIOLATION": 1
}

df["Crime_Severity_Score"] = df["Primary Type"].map(severity_map)

In [9]:
# Categorical Encoding for Crime type and Location Description

# Frequency encoding for Location Description and crime type
freq = df["Location Description"].value_counts(normalize=True)
df["Location_Desc_freq"] = df["Location Description"].map(freq)

freq = df["Primary Type"].value_counts(normalize=True)
df["Primary Type_freq"] = df["Primary Type"].map(freq)

In [10]:
# Normalizing location coordinates
scaler = StandardScaler()

df[["Latitude_Norm", "Longitude_Norm"]] = scaler.fit_transform(
    df[["Latitude", "Longitude"]]
)

In [11]:
df['Arrest'] = df['Arrest'].astype(int)
df['Domestic'] = df['Domestic'].astype(int)

In [12]:
encoder = LabelEncoder()
df['Primary Type'] = encoder.fit_transform(df['Primary Type'])
df['Location Description'] = encoder.fit_transform(df['Location Description'])
df['Day_of_Week'] = df['Day_of_Week'].replace({'Monday':0, 'Tuesday':1, 'Wednesday':2, 'Thursday':'3', 'Friday':4, 'Saturday':5, 'Sunday':6}).astype(int)
df['Season'] = encoder.fit_transform(df['Season'])

In [13]:
df.head()

,Date,Primary Type,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,Year,Latitude,Longitude,Month,Day_of_Week,Hour,Is_Weekend,Season,Lat_bin,Lon_bin,Crime_Severity_Score,Location_Desc_freq,Primary Type_freq,Latitude_Norm,Longitude_Norm
0,2016-12-31 23:59:00,30,3,0,0,1651,16,41,76,2016,41.976290,-87.905227,12,5,23,1,3,"(41.753, 42.023]","(-87.941, -87.733]",1.0,0.000122,0.216753,1.470472,-3.629450
1,2016-12-31 23:58:00,2,135,0,0,522,5,34,49,2016,41.688033,-87.623931,12,5,23,1,3,"(41.482, 41.753]","(-87.733, -87.525]",3.0,0.163276,0.178889,-1.663659,0.753301
2,2016-12-31 23:55:00,16,135,1,0,1932,19,32,6,2016,41.936885,-87.664770,12,5,23,1,3,"(41.753, 42.023]","(-87.733, -87.525]",1.0,0.163276,0.001666,1.042028,0.117016
3,2016-12-31 23:55:00,2,99,0,1,111,1,42,32,2016,41.886815,-87.625593,12,5,23,1,3,"(41.753, 42.023]","(-87.733, -87.525]",3.0,0.003961,0.178889,0.497632,0.727418
4,2016-12-31 23:54:00,6,135,0,1,623,6,6,69,2016,41.752307,-87.619798,12,5,23,1,3,"(41.482, 41.753]","(-87.733, -87.525]",2.0,0.163276,0.111719,-0.964830,0.817709


In [14]:
df.shape

(2965087, 24)

In [19]:
df['Arrest'].unique()[:30]

array([0, 1])

In [20]:
df['Domestic'].unique()[:30]

array([0, 1])

In [21]:
df.to_pickle('engineered_chicago_crime_data.pkl')